In [1]:
pip install pandasql

  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26773 sha256=39ed8dedb368785ea65c00bb1cf9de1e6756d45848af2fd61fb39bf47b71edad
  Stored in directory: /root/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql


In [2]:
import pandas as pd
from pandasql import sqldf

In [3]:
pysqldf = lambda q: sqldf(q, globals())


In [4]:
buyers = pd.read_csv('/content/buyers.csv', sep=',')
order_items = pd.read_csv('/content/order_items.csv', sep=',')
orders = pd.read_csv('/content/orders.csv', sep=',')
payments = pd.read_csv('/content/payments.csv', sep=',')
products = pd.read_csv('/content/products.csv', sep=',')
sellers = pd.read_csv('/content/sellers.csv', sep=',')

In [5]:
display(buyers.head())
display(order_items.head())
display(orders.head())
display(payments.head())
display(products.head())
display(sellers.head())


,id,name,city,state,segment,created_at
0,1,Nascimento & Costa Distribuidora Mercearia,Recife,DF,supermarket,2023-04-03 03:46:36
1,2,Lima & Almeida Atacado Mercearia,Rio de Janeiro,PE,convenience,2023-08-05 01:36:12
2,3,Ferreira & Costa Grupo Mercearia,São Paulo,PR,grocery,2023-07-10 11:37:01
3,4,Lima & Almeida Comércio Supermercado,Campo Grande,MT,convenience,2023-10-13 01:08:52
4,5,Ferreira & Ferreira Comércio Mercearia,São Paulo,PB,convenience,2024-04-20 00:23:27


,id,order_id,product_id,qty,unit_price,discount
0,1,1,500,6,240.06,117.48
1,2,1,778,6,126.47,94.54
2,3,2,346,39,492.08,98.86
3,4,2,248,20,278.55,588.33
4,5,3,701,9,298.75,272.33


,id,seller_id,buyer_id,status,created_at,total_value
0,1,114,2806,delivered,2023-08-22 05:25:59,1987.16
1,2,90,2586,processing,2024-07-19 03:57:54,24074.93
2,3,96,1849,completed,2024-06-14 22:16:15,2416.42
3,4,10,116,processing,2024-07-11 04:46:30,3871.48
4,5,16,2195,delivered,2023-09-07 20:06:40,15367.65


,id,order_id,paid_at,amount,method,status
0,1,1,2023-08-23 19:25:59,1987.16,transfer,paid
1,2,2,2024-07-20 17:57:54,24074.93,boleto,paid
2,3,3,2024-06-16 10:16:15,2416.42,transfer,paid
3,4,4,2024-07-11 11:46:30,3871.48,boleto,paid
4,5,5,2023-09-08 00:06:40,15367.65,transfer,paid


,id,name,category,seller_id,active,unit_cost
0,1,Produto Laticínios Linha 1,Snacks,31,1,104.39
1,2,Produto Grãos Linha 2,Carnes,71,1,153.19
2,3,Produto Enlatados Linha 3,Grãos,9,0,90.57
3,4,Produto Snacks Linha 4,Bebidas,13,1,158.39
4,5,Produto Carnes Linha 5,Grãos,58,1,124.43


,id,name,state,plan,created_at
0,1,Santos & Silva Atacado Distribuidora,BA,free,2023-04-17 00:16:11
1,2,Souza & Souza Comércio Distribuidora,DF,premium,2022-09-09 22:06:21
2,3,Santos & Almeida Distribuidora Distribuidora,AM,basic,2022-12-16 08:31:25
3,4,Nascimento & Ferreira Alimentos Distribuidora,GO,basic,2023-05-30 23:12:37
4,5,Silva & Santos Suprimentos Distribuidora,BA,premium,2023-03-06 10:29:33


# **Resolução dos Desafios 1 a 4**

Desafio 1 - Faturamento Mensal

A última data disponível na base (29/11/2024) foi utilizada como referência
para a janela dos últimos 12 meses. O faturamento bruto foi calculado com base
em qty × unit_price, antes da aplicação dos descontos.

In [6]:
# Desafio 1 — Faturamento Mensal

query = """
SELECT
    strftime('%Y-%m', o.created_at) AS mes,
    ROUND(SUM(oi.qty * oi.unit_price), 2) AS faturamento_bruto,
    COUNT(DISTINCT o.id) AS quantidade_pedidos,
    ROUND(
        SUM(oi.qty * oi.unit_price)
        / NULLIF(COUNT(DISTINCT o.id), 0),
        2
    ) AS ticket_medio
FROM orders AS o
INNER JOIN order_items AS oi
    ON o.id = oi.order_id
WHERE o.status IN ('completed', 'delivered')
  AND o.created_at >= (
      SELECT datetime(
          strftime('%Y-%m-01', MAX(created_at)),
          '-11 months'
      )
      FROM orders
  )
GROUP BY strftime('%Y-%m', o.created_at)
ORDER BY mes DESC;
"""

resultado_desafio1 = pysqldf(query)

# Formatação do resultado para melhor visualização

resultado_formatado = resultado_desafio1.copy()

resultado_formatado['faturamento_bruto'] = resultado_formatado[
    'faturamento_bruto'
].map(
    lambda x: f"R$ {x:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')
)

resultado_formatado['ticket_medio'] = resultado_formatado[
    'ticket_medio'
].map(
    lambda x: f"R$ {x:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')
)

resultado_formatado['quantidade_pedidos'] = resultado_formatado[
    'quantidade_pedidos'
].map(
    lambda x: f"{x:,}".replace(',', '.')
)

display(resultado_formatado)

,mes,faturamento_bruto,quantidade_pedidos,ticket_medio
0,2024-11,"R$ 61.906.773,58",3.643,"R$ 16.993,35"
1,2024-10,"R$ 68.259.702,23",3.863,"R$ 17.670,13"
2,2024-09,"R$ 65.922.278,93",3.779,"R$ 17.444,37"
3,2024-08,"R$ 65.839.948,65",3.743,"R$ 17.590,15"
4,2024-07,"R$ 65.286.755,51",3.862,"R$ 16.904,91"
5,2024-06,"R$ 64.151.214,87",3.644,"R$ 17.604,61"
6,2024-05,"R$ 66.388.644,41",3.848,"R$ 17.252,77"
7,2024-04,"R$ 63.237.591,49",3.653,"R$ 17.311,14"
8,2024-03,"R$ 67.235.984,93",3.877,"R$ 17.342,27"
9,2024-02,"R$ 63.193.936,16",3.628,"R$ 17.418,39"


Desafio 2 - Crescimento de GMV

A consulta consolida o GMV e a quantidade de pedidos por seller nos dois trimestres analisados. Em seguida, compara os períodos, aplica o mínimo de 50 pedidos em ambos e calcula o percentual de crescimento, retornando os 10 maiores resultados. Observação: o enunciado não especifica o período de referência. Foi considerado como trimestre atual o período mais recente disponível na base T4/2024 e como trimestre anterior o T3/2024.

In [7]:
# Desafio 2

query = """
WITH vendas_trimestre AS (
    SELECT
        o.seller_id,
        CASE
            WHEN strftime('%m', o.created_at) BETWEEN '07' AND '09' THEN 'Q3'
            WHEN strftime('%m', o.created_at) BETWEEN '10' AND '12' THEN 'Q4'
        END AS trimestre,
        COUNT(DISTINCT o.id) AS quantidade_pedidos,
        SUM(oi.qty * oi.unit_price) AS gmv
    FROM orders AS o
    INNER JOIN order_items AS oi
        ON o.id = oi.order_id
    WHERE o.status IN ('completed', 'delivered')
      AND o.created_at >= '2024-07-01'
      AND o.created_at < '2025-01-01'
    GROUP BY
        o.seller_id,
        trimestre
),

comparacao AS (
    SELECT
        seller_id,
        MAX(CASE WHEN trimestre = 'Q3' THEN quantidade_pedidos END) AS pedidos_q3,
        MAX(CASE WHEN trimestre = 'Q4' THEN quantidade_pedidos END) AS pedidos_q4,
        MAX(CASE WHEN trimestre = 'Q3' THEN gmv END) AS gmv_q3,
        MAX(CASE WHEN trimestre = 'Q4' THEN gmv END) AS gmv_q4
    FROM vendas_trimestre
    GROUP BY seller_id
)

SELECT
    s.name AS seller,
    s.state AS estado,
    ROUND(c.gmv_q3, 2) AS gmv_trimestre_anterior,
    ROUND(c.gmv_q4, 2) AS gmv_trimestre_atual,
    ROUND(
        ((c.gmv_q4 - c.gmv_q3) / NULLIF(c.gmv_q3, 0)) * 100,
        2
    ) AS percentual_crescimento
FROM comparacao AS c
INNER JOIN sellers AS s
    ON s.id = c.seller_id
WHERE c.pedidos_q3 >= 50
  AND c.pedidos_q4 >= 50
ORDER BY percentual_crescimento DESC
LIMIT 10;
"""

resultado_desafio2 = pysqldf(query)

# Formatação do resultado para melhor visualização

resultado_formatado = resultado_desafio2.copy()

resultado_formatado['gmv_trimestre_anterior'] = resultado_formatado[
    'gmv_trimestre_anterior'
].map(
    lambda x: f"R$ {x:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')
)

resultado_formatado['gmv_trimestre_atual'] = resultado_formatado[
    'gmv_trimestre_atual'
].map(
    lambda x: f"R$ {x:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')
)

resultado_formatado['percentual_crescimento'] = resultado_formatado[
    'percentual_crescimento'
].map(
    lambda x: f"{x:.2f}%".replace('.', ',')
)

display(resultado_formatado)

,seller,estado,gmv_trimestre_anterior,gmv_trimestre_atual,percentual_crescimento
0,Rodrigues & Almeida Atacado Distribuidora,MG,"R$ 947.229,03","R$ 922.906,69","-2,57%"
1,Costa & Silva Alimentos Distribuidora,BA,"R$ 1.094.018,29","R$ 1.063.689,63","-2,77%"
2,Costa & Santos Suprimentos Distribuidora,DF,"R$ 946.849,42","R$ 872.947,94","-7,80%"
3,Nascimento & Souza Alimentos Distribuidora,SP,"R$ 1.117.256,71","R$ 1.019.379,83","-8,76%"
4,Almeida & Almeida Alimentos Distribuidora,RS,"R$ 1.109.916,83","R$ 997.166,12","-10,16%"
5,Almeida & Souza Mercado Distribuidora,PR,"R$ 1.187.852,14","R$ 1.055.798,97","-11,12%"
6,Lima & Almeida Grupo Distribuidora,CE,"R$ 1.133.549,02","R$ 1.007.041,67","-11,16%"
7,Nascimento & Almeida Alimentos Distribuidora,SP,"R$ 1.027.109,41","R$ 909.132,24","-11,49%"
8,Costa & Santos Atacado Distribuidora,MG,"R$ 1.191.560,72","R$ 1.043.909,09","-12,39%"
9,Almeida & Rodrigues Distribuidora Distribuidora,RJ,"R$ 1.190.506,71","R$ 982.087,64","-17,51%"


Desafio 3 - Descontos Abusivos


O valor bruto e o desconto total foram calculados por pedido. Após excluir pedidos cancelados, foram selecionados aqueles cujo desconto representa mais de 40% do valor bruto, identificando também o seller responsável e a data do pedido.

In [ ]:
# Desafio 3 — Descontos Abusivos

query = """
WITH pedidos AS (
    SELECT
        o.id AS order_id,
        o.seller_id,
        o.created_at,
        ROUND(SUM(oi.qty * oi.unit_price), 2) AS valor_bruto,
        ROUND(SUM(oi.discount), 2) AS desconto_total
    FROM orders AS o
    INNER JOIN order_items AS oi
        ON o.id = oi.order_id
    WHERE o.status <> 'cancelled'
    GROUP BY
        o.id,
        o.seller_id,
        o.created_at
)

SELECT
    p.order_id,
    s.name AS seller,
    p.created_at,
    p.valor_bruto,
    p.desconto_total,
    ROUND(
        (p.desconto_total / NULLIF(p.valor_bruto, 0)) * 100,
        2
    ) AS percentual_desconto
FROM pedidos AS p
INNER JOIN sellers AS s
    ON s.id = p.seller_id
WHERE p.desconto_total / NULLIF(p.valor_bruto, 0) > 0.40
ORDER BY percentual_desconto DESC;
"""

resultado_desafio3 = pysqldf(query)

# Formatação do resultado para melhor visualização

resultado_formatado = resultado_desafio3.copy()

resultado_formatado['valor_bruto'] = resultado_formatado['valor_bruto'].map(
    lambda x: f"R$ {x:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')
)

resultado_formatado['desconto_total'] = resultado_formatado['desconto_total'].map(
    lambda x: f"R$ {x:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')
)

resultado_formatado['percentual_desconto'] = resultado_formatado['percentual_desconto'].map(
    lambda x: f"{x:.2f}%".replace('.', ',')
)

display(resultado_formatado)

,order_id,seller,created_at,valor_bruto,desconto_total,percentual_desconto
0,72401,Costa & Oliveira Atacado Distribuidora,2024-01-21 02:43:43,"R$ 930,30","R$ 558,02","59,98%"
1,42513,Costa & Costa Atacado Distribuidora,2024-03-27 02:30:57,"R$ 4.082,60","R$ 2.447,96","59,96%"
2,36119,Costa & Costa Atacado Distribuidora,2024-11-14 01:05:01,"R$ 21.030,24","R$ 12.598,00","59,90%"
3,53223,Costa & Costa Atacado Distribuidora,2024-05-06 01:56:47,"R$ 7.970,40","R$ 4.773,99","59,90%"
4,71165,Santos & Almeida Suprimentos Distribuidora,2024-11-09 02:32:22,"R$ 13.385,60","R$ 8.017,88","59,90%"
...,...,...,...,...,...,...
896,38595,Costa & Costa Atacado Distribuidora,2023-08-21 04:52:02,"R$ 27.785,26","R$ 11.124,91","40,04%"
897,64306,Oliveira & Santos Distribuidora Distribuidora,2024-10-12 21:22:52,"R$ 6.907,95","R$ 2.765,81","40,04%"
898,32057,Souza & Ferreira Alimentos Distribuidora,2024-03-10 03:51:38,"R$ 24.658,56","R$ 9.868,74","40,02%"
899,59315,Santos & Rodrigues Distribuidora Distribuidora,2023-08-10 22:52:09,"R$ 21.647,49","R$ 8.661,37","40,01%"


Desafio 4 - Produtos com comportamento estranho

Foi utilizado o row_number para identificar o maior valor unitário por pedido, com validações prévias dos critérios do desafio.

In [9]:
# Desafio 4 — Produtos com comportamento estranho

query = """
WITH itens_classificados AS (
    SELECT
        oi.order_id,
        oi.product_id,
        oi.qty,
        oi.unit_price,
        ROW_NUMBER() OVER (
            PARTITION BY oi.order_id
            ORDER BY oi.unit_price DESC
        ) AS ranking_item
    FROM order_items AS oi
),

vendas_produto AS (
    SELECT
        product_id,
        SUM(qty) AS unidades_vendidas
    FROM order_items
    GROUP BY product_id
),

produtos_maior_valor AS (
    SELECT DISTINCT
        product_id
    FROM itens_classificados
    WHERE ranking_item = 1
)

SELECT
    p.id AS product_id,
    p.name AS produto,
    vp.unidades_vendidas
FROM products AS p
INNER JOIN vendas_produto AS vp
    ON p.id = vp.product_id
LEFT JOIN produtos_maior_valor AS pmv
    ON p.id = pmv.product_id
WHERE vp.unidades_vendidas > 1000
  AND pmv.product_id IS NULL
ORDER BY vp.unidades_vendidas DESC;
"""

resultado_desafio4 = pysqldf(query)
display(resultado_desafio4)

,product_id,produto,unidades_vendidas


Validação dos critérios

---


1.   Mais de 1.000 unidades vendidas: 800 produtos.
2.   Já foi o maior valor unitário em algum pedido: 800 produtos.
3.   Produtos que atendem aos dois critérios: 0.

Conclusão

---


Identifiquei 800 produtos com mais de 1.000 unidades vendidas. Ao validar o segundo critério, constatei que todos os 800 já foram o item de maior valor unitário em pelo menos um pedido. Portanto, não identifiquei nenhum produto que atenda simultaneamente aos critérios definidos.

In [ ]:
# Consultas feitas para validação da conclusão
# Produtos com mais de 1.000 unidades - 1 critério
query = """
SELECT
    product_id,
    SUM(qty) AS unidades_vendidas
FROM order_items
GROUP BY product_id
HAVING SUM(qty) > 1000
ORDER BY unidades_vendidas DESC;
"""

resultado = pysqldf(query)
display(resultado)
# Produtos que já foram o maior valor unitário -  2 critério
query = """
WITH itens_classificados AS (
    SELECT
        order_id,
        product_id,
        unit_price,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY unit_price DESC
        ) AS ranking_item
    FROM order_items
)

SELECT DISTINCT
    product_id
FROM itens_classificados
WHERE ranking_item = 1
ORDER BY product_id;
"""

resultado = pysqldf(query)
display(resultado)

# A confirmação da minha analise
query = """
WITH itens_classificados AS (
    SELECT
        order_id,
        product_id,
        unit_price,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY unit_price DESC
        ) AS ranking_item
    FROM order_items
),

produtos_maior_valor AS (
    SELECT DISTINCT
        product_id
    FROM itens_classificados
    WHERE ranking_item = 1
),

vendas_produto AS (
    SELECT
        product_id,
        SUM(qty) AS unidades_vendidas
    FROM order_items
    GROUP BY product_id
)

SELECT
    COUNT(*) AS produtos_acima_1000,
    COUNT(pmv.product_id) AS produtos_que_ja_foram_maior_valor,
    COUNT(*) - COUNT(pmv.product_id) AS produtos_elegiveis
FROM vendas_produto AS vp
LEFT JOIN produtos_maior_valor AS pmv
    ON vp.product_id = pmv.product_id
WHERE vp.unidades_vendidas > 1000;
"""

resultado = pysqldf(query)
display(resultado)

# **Análise Exploratória — Entendimento dos Dados**

**OBS**: As análises apresentadas nesta seção têm caráter exploratório e servem apenas para entendimento e validação inicial da base, não compondo diretamente as respostas dos desafios.

In [ ]:
# Quantidade de registros
print("buyers:", len(buyers))
print("order_items:", len(order_items))
print("orders:", len(orders))
print("payments:", len(payments))
print("products:", len(products))
print("sellers:", len(sellers))

In [ ]:
# Status dos pedidos
query = """
SELECT
    status,
    COUNT(*) AS quantidade_pedidos
FROM orders
GROUP BY status
ORDER BY quantidade_pedidos DESC
"""

resultado_desafio1 = pysqldf(query)
display(resultado_desafio1)

In [ ]:
# Período dos dados
query = """
SELECT
    MIN(created_at) AS primeira_data,
    MAX(created_at) AS ultima_data
FROM orders
"""

resultado = pysqldf(query)
display(resultado)

In [ ]:
# valor dos pedidos
query = """
SELECT
    o.id AS order_id,
    o.total_value,
    SUM(oi.qty * oi.unit_price) AS valor_itens,
    SUM(oi.discount) AS desconto_total
FROM orders o
JOIN order_items oi
    ON o.id = oi.order_id
GROUP BY
    o.id,
    o.total_value
LIMIT 10
"""

resultado = pysqldf(query)
display(resultado)

In [ ]:
# Pedidos por trimestre
query = """
SELECT
    strftime('%Y', created_at) AS ano,
    ((CAST(strftime('%m', created_at) AS INTEGER) - 1) / 3) + 1 AS trimestre,
    COUNT(*) AS quantidade_pedidos
FROM orders
GROUP BY
    strftime('%Y', created_at),
    ((CAST(strftime('%m', created_at) AS INTEGER) - 1) / 3) + 1
ORDER BY
    ano,
    trimestre
"""

resultado = pysqldf(query)
display(resultado)

In [ ]:
# Quantidade de pedidos por seller
query = """
SELECT
    o.seller_id,
    CASE
        WHEN strftime('%m', o.created_at) BETWEEN '07' AND '09' THEN 'Q3'
        WHEN strftime('%m', o.created_at) BETWEEN '10' AND '12' THEN 'Q4'
    END AS trimestre,
    COUNT(DISTINCT o.id) AS quantidade_pedidos
FROM orders AS o
WHERE o.status IN ('completed', 'delivered')
  AND o.created_at >= '2024-07-01'
  AND o.created_at < '2025-01-01'
GROUP BY
    o.seller_id,
    trimestre
ORDER BY
    o.seller_id,
    trimestre;
"""

resultado = pysqldf(query)
display(resultado)

In [ ]:
# Validação do crescimento de GMV dos sellers
query = """
WITH vendas_trimestre AS (
    SELECT
        o.seller_id,
        CASE
            WHEN strftime('%m', o.created_at) BETWEEN '07' AND '09' THEN 'Q3'
            ELSE 'Q4'
        END AS trimestre,
        COUNT(DISTINCT o.id) AS quantidade_pedidos,
        SUM(oi.qty * oi.unit_price) AS gmv
    FROM orders AS o
    INNER JOIN order_items AS oi
        ON o.id = oi.order_id
    WHERE o.status IN ('completed', 'delivered')
      AND o.created_at >= '2024-07-01'
      AND o.created_at < '2025-01-01'
    GROUP BY o.seller_id, trimestre
),

comparacao AS (
    SELECT
        seller_id,
        MAX(CASE WHEN trimestre = 'Q3' THEN quantidade_pedidos END) AS pedidos_q3,
        MAX(CASE WHEN trimestre = 'Q4' THEN quantidade_pedidos END) AS pedidos_q4,
        MAX(CASE WHEN trimestre = 'Q3' THEN gmv END) AS gmv_q3,
        MAX(CASE WHEN trimestre = 'Q4' THEN gmv END) AS gmv_q4
    FROM vendas_trimestre
    GROUP BY seller_id
)

SELECT
    COUNT(*) AS sellers_elegiveis,
    SUM(CASE WHEN gmv_q4 > gmv_q3 THEN 1 ELSE 0 END) AS sellers_com_crescimento,
    SUM(CASE WHEN gmv_q4 < gmv_q3 THEN 1 ELSE 0 END) AS sellers_com_queda
FROM comparacao
WHERE pedidos_q3 >= 50
  AND pedidos_q4 >= 50;
"""

resultado = pysqldf(query)
display(resultado)

In [ ]:
# Validar a composição do valor dos pedidos
query = """
SELECT
    o.id AS order_id,
    o.status,
    o.seller_id,
    o.created_at,
    ROUND(SUM(oi.qty * oi.unit_price), 2) AS valor_bruto,
    ROUND(SUM(oi.discount), 2) AS desconto_total,
    ROUND(o.total_value, 2) AS valor_total
FROM orders AS o
INNER JOIN order_items AS oi
    ON o.id = oi.order_id
GROUP BY
    o.id,
    o.status,
    o.seller_id,
    o.created_at,
    o.total_value
LIMIT 10;
"""

resultado = pysqldf(query)
display(resultado)

In [ ]:
# verificação do ROW_NUMBER
query = """
SELECT
    order_id,
    product_id,
    unit_price,
    ROW_NUMBER() OVER (
        PARTITION BY order_id
        ORDER BY unit_price DESC
    ) AS ranking_item
FROM order_items
LIMIT 20;
"""

teste = pysqldf(query)
display(teste)

In [ ]:
# verifição de produto com mais de 1000 vendidas
query = """
SELECT
    product_id,
    SUM(qty) AS unidades_vendidas
FROM order_items
GROUP BY product_id
HAVING SUM(qty) > 1000
ORDER BY unidades_vendidas DESC;
"""

resultado = pysqldf(query)
display(resultado)

,product_id,unidades_vendidas
0,415,8793
1,767,8486
2,528,8413
3,794,8311
4,625,8123
...,...,...
795,133,5571
796,620,5532
797,132,5488
798,507,5434


In [ ]:
# Verificação dos produtos
query = """
WITH itens_classificados AS (
    SELECT
        order_id,
        product_id,
        unit_price,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY unit_price DESC
        ) AS ranking_item
    FROM order_items
)

SELECT DISTINCT
    product_id
FROM itens_classificados
WHERE ranking_item = 1
ORDER BY product_id;
"""

resultado = pysqldf(query)
display(resultado)

,product_id
0,1
1,2
2,3
3,4
4,5
...,...
795,796
796,797
797,798
798,799


In [ ]:
# Validação

query = """
WITH itens_classificados AS (
    SELECT
        order_id,
        product_id,
        unit_price,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY unit_price DESC
        ) AS ranking_item
    FROM order_items
),

produtos_maior_valor AS (
    SELECT DISTINCT
        product_id
    FROM itens_classificados
    WHERE ranking_item = 1
),

vendas_produto AS (
    SELECT
        product_id,
        SUM(qty) AS unidades_vendidas
    FROM order_items
    GROUP BY product_id
)

SELECT
    COUNT(*) AS produtos_acima_1000,
    COUNT(pmv.product_id) AS produtos_que_ja_foram_maior_valor,
    COUNT(*) - COUNT(pmv.product_id) AS produtos_elegiveis
FROM vendas_produto AS vp
LEFT JOIN produtos_maior_valor AS pmv
    ON vp.product_id = pmv.product_id
WHERE vp.unidades_vendidas > 1000;
"""

resultado = pysqldf(query)
display(resultado)

,produtos_acima_1000,produtos_que_ja_foram_maior_valor,produtos_elegiveis
0,800,800,0
